# Rasa Demo Bot Analytics Walkthrough

This notebook demonstrates how to generate simulated Rasa conversation logs, analyse fallback events, and build dashboard-ready visualisations. It is designed to be run in Google Colab and relies only on open-source libraries available via `pip`.

In [ ]:
# If running in a fresh Google Colab session, install the required packages
%pip install --quiet pandas numpy matplotlib seaborn plotly

In [ ]:
import json
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style="whitegrid")

## 1. Simulate Rasa Conversation Logs
The helper below creates a realistic-looking log dataset compatible with Rasa tracker exports. Each row represents a user message or bot action annotated with metadata that can be aggregated for analytics.

In [ ]:
def simulate_rasa_logs(num_sessions: int = 150, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    base_time = datetime.now() - timedelta(days=7)
    intents = ["greet", "inform_plant_issue", "ask_for_recommendation", "affirm", "goodbye", "out_of_scope"]
    channels = ["webchat", "telegram", "whatsapp"]
    steps = []
    for session_id in range(1, num_sessions + 1):
        start_time = base_time + timedelta(minutes=int(rng.uniform(0, 60 * 24 * 7)))
        num_turns = rng.integers(4, 12)
        channel = rng.choice(channels, p=[0.5, 0.2, 0.3])
        fallback_probability = 0.05 + 0.15 * (channel == "webchat")
        journey_state = "in_progress"
        for turn in range(num_turns):
            timestamp = start_time + timedelta(minutes=turn * rng.uniform(0.5, 3.0))
            intent = rng.choice(intents, p=[0.18, 0.28, 0.18, 0.1, 0.16, 0.1])
            is_fallback = rng.random() < fallback_probability if intent == "out_of_scope" else False
            if is_fallback:
                journey_state = "fallback"
            elif intent == "goodbye":
                journey_state = "resolved"
            steps.append({
                "session_id": f"S{session_id:04d}",
                "timestamp": timestamp,
                "intent": intent,
                "channel": channel,
                "is_fallback": is_fallback,
                "journey_state": journey_state
            })
    df = pd.DataFrame(steps).sort_values("timestamp").reset_index(drop=True)
    return df

logs_df = simulate_rasa_logs()
logs_df.head()

## 2. Fallback Optimisation Analytics
We focus on fallback analysis to improve recovery flows. The metric suite below tracks fallback frequency by channel and identifies high-risk intents.

In [ ]:
fallback_summary = (
    logs_df.groupby("channel")["is_fallback"].mean().rename("fallback_rate")
    .reset_index()
)
fallback_summary["fallback_rate"] = fallback_summary["fallback_rate"].round(3)
fallback_summary

In [ ]:
fig = px.bar(
    fallback_summary,
    x="channel",
    y="fallback_rate",
    title="Fallback Rate by Channel",
    text_auto=".1%"
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [ ]:
intent_fallback = (
    logs_df.groupby("intent")["is_fallback"].mean().rename("fallback_rate")
    .reset_index()
)
intent_fallback.sort_values("fallback_rate", ascending=False).head(10)

### Session-Level Heatmap
This heatmap helps identify when fallback events occur in the conversation timeline. The darker the colour, the later the fallback.

In [ ]:
session_pivots = (
    logs_df.assign(step=lambda df: df.groupby("session_id").cumcount())
    .pivot_table(
        index="session_id",
        columns="step",
        values="is_fallback",
        aggfunc="max",
        fill_value=0
    )
)
plt.figure(figsize=(10, 6))
sns.heatmap(session_pivots.iloc[:40], cmap="YlOrRd", cbar_kws={"label": "Fallback Occurrence"})
plt.title("Fallback Occurrence Heatmap (first 40 sessions)")
plt.xlabel("Turn Index")
plt.ylabel("Session")
plt.tight_layout()
plt.show()

## 3. Dashboard-Ready Aggregations
The following cells derive metrics for a chatbot performance dashboard covering platform performance, user journey attribution, and feedback signals.

In [ ]:
top_intents = (
    logs_df.groupby("intent").size().sort_values(ascending=False).reset_index(name="count")
)
px.bar(top_intents, x="intent", y="count", title="Top User Intents").show()

In [ ]:
journey_summary = (
    logs_df.groupby(["session_id"]).agg(
        channel=("channel", "first"),
        resolved=("journey_state", lambda s: (s == "resolved").any()),
        fallback=("is_fallback", "any")
    )
)
journey_counts = {
    "Start": len(journey_summary),
    "Resolution": int(journey_summary["resolved"].sum()),
    "Fallback Exit": int((~journey_summary["resolved"] & journey_summary["fallback"]).sum())
}
funnel_fig = go.Figure(go.Funnel(
    y=list(journey_counts.keys()),
    x=list(journey_counts.values()),
    textinfo="value+percent initial"
))
funnel_fig.update_layout(title="User Journey Funnel")
funnel_fig.show()

In [ ]:
feedback_counts = pd.DataFrame({
    "outcome": ["Successful", "Fallback"],
    "count": [journey_counts["Resolution"], journey_counts["Fallback Exit"]]
})
px.pie(feedback_counts, names="outcome", values="count", title="Conversation Outcomes", hole=0.3).show()

In [ ]:
sessions_over_time = (
    logs_df.groupby(pd.Grouper(key="timestamp", freq="6H"))["session_id"].nunique().reset_index(name="sessions")
)
px.line(sessions_over_time, x="timestamp", y="sessions", title="Sessions over Time").show()

## 4. Export Aggregated Metrics
These tables can be sent to downstream BI tools or dashboards.

In [ ]:
metrics_bundle = {
    "fallback_summary": fallback_summary.to_dict(orient="records"),
    "intent_counts": top_intents.to_dict(orient="records"),
    "journey_counts": journey_counts,
}
with open("rasa_demo_metrics.json", "w") as f:
    json.dump(metrics_bundle, f, default=str, indent=2)
metrics_bundle